# FracAtlas sweep runner (Kaggle)

Runs any `configs/sweep_*.yaml` (variants x seeds, one job queue per GPU), then the
validation comparison and its pre-registered decision. All logic is in the repo.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, *Add Input* -> the FracAtlas dataset.
Run with **Save Version -> Save & Run All** so it finishes in the background.

In [ ]:
SWEEP = "configs/sweep_phase5.yaml"  # which sweep to run
SMOKE = False  # True: 1-epoch run on a tiny subset into /kaggle/working/smoke (~5 min)

In [ ]:
import os
import subprocess

import yaml

WORKDIR = "/kaggle/working/repo"
if not os.path.isdir(WORKDIR):
    url = "https://github.com/Akinchan-Jain-7/fracatlas-fracture-detection.git"
    subprocess.run(["git", "clone", "-q", url, WORKDIR], check=True)
os.chdir(WORKDIR)
os.environ["FRACDET_DATA_DIR"] = "/kaggle/input"
with open(SWEEP) as f:
    NAME = yaml.safe_load(f)["name"]
!git log --oneline -1
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; nproc

In [ ]:
if SMOKE:
    !python scripts/sweep.py {SWEEP} --gpus 0,1 --out-root /kaggle/working/smoke --set train.num_workers=2 train.epochs=1 debug.n_train=128 debug.n_val=64 paths.output_dir=/kaggle/working/smoke
    !python scripts/compare_runs.py {SWEEP} --out /kaggle/working/smoke/{NAME}
else:
    !python scripts/sweep.py {SWEEP} --gpus 0,1 --set train.num_workers=2

In [ ]:
if not SMOKE:
    !python scripts/compare_runs.py {SWEEP}
    from IPython.display import Image, display

    display(Image(f"outputs/sweeps/{NAME}/compare.png"))
    # keep best.pt (a winner gets a test evaluation); drop last.pt to save space
    !rm -f outputs/runs/*/last.pt && du -sh outputs